# Spatial perturbation benchmark — run
Runs the 2x2 (seed x propagation) benchmark, prints the metric tables, and draws figures.
Run on the lab server (env `spatial-tumor-ai`) so it can read the real .h5mu.

In [ ]:
import matplotlib
%matplotlib inline
from spbench.adapters import get_adapter
from spbench.config import run_benchmark
from spbench.viz import plot_2x2, plot_attribution
import numpy as np, pandas as pd

## 1. Load data via adapter (swap adapter/path here to change dataset)

In [ ]:
DIR='/home/yiru/database/spatial_perturbed_processed/CRISPR_based/Saunders_2025_40513557'
data = get_adapter('saunders')(DIR, max_files=4).load()
print(data.n_cells, data.n_genes, 'perturbations:', len(data.perturbations()))

## 2. Run the benchmark (trivial seed + Gaussian + GCN)

In [ ]:
PERTS = data.perturbations()[:10]   # later: replace with the 14 MC-spatial significant
res = run_benchmark(data, perturbations=PERTS, k=15, k_ref=5,
                    gcn_kwargs={'hidden':64,'epochs':30})

## 3. Metric table (per-perturbation 2x2 + attribution + leakage)

In [ ]:
rows=[]
for p in PERTS:
    g=res['grids'][p]; a=res['attribution'][p]
    rows.append(dict(perturbation=p, e1=g['1']['energy_prop'], e2=g['2']['energy_prop'],
                     e3=g['3']['energy_prop'], e4=g['4']['energy_prop'],
                     seed_cost=a['seed_cost'], learned_value=a['learned_value'],
                     end_to_end=a['end_to_end'], leak_ok=res['leakage_pass'][p]))
df=pd.DataFrame(rows).sort_values('end_to_end'); df

## 4. Figures

In [ ]:
fig=plot_2x2(res['grids'][PERTS[0]], title=PERTS[0]); fig.savefig('fig_2x2.png', dpi=150)

In [ ]:
fig=plot_attribution(res['attribution']); fig.savefig('fig_attribution.png', dpi=150)

## 5. Ranking + go/no-go note
Lowest end-to-end E-distance = best. Add the rho_niche (with vs without niche) here once the
no-niche variant is wired, and check the +0.10 gate.

In [ ]:
print('ranking (best first):', res['ranking'])